In [3]:
import os
import glob
import numpy as np
from pathlib import Path
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
import astrometry
import hipercam as hcam

def solve_hcm_direct(hcm_file, ccd_label='1', win_label='1',
                     scale_low=0.1, scale_high=2.0,
                     ra_center=None, dec_center=None, radius=10.0,
                     astrometry_cache='/lustre/MSSP/sittipong/astrometry_cache'):
    """
    อ่านไฟล์ .hcm สกัดดาว และทำ Plate Solving โดยโหลด Index Files ตรงจากโฟลเดอร์
    เพื่อแก้ปัญหา KeyError และข้อจำกัดเรื่อง Scale
    """
    print(f"\n[1] Reading file: {hcm_file} (CCD: {ccd_label}, Win: {win_label})")
    
    # ── 1. เปิดไฟล์ภาพ .hcm และดึงข้อมูลพิกเซล ──
    try:
        mccd = hcam.MCCD.read(hcm_file)
        window = mccd[ccd_label][win_label]
        data = window.data
        xbin, ybin = window.xbin, window.ybin
        h, w = data.shape
    except Exception as e:
        print(f"❌ Error reading HCM file: {e}")
        return None

    # ── 2. ค้นหาดาวด้วย DAOStarFinder ──
    print("[2] Finding stars using photutils (DAOStarFinder)...")
    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    
    # ใช้ fwhm=4.0 และกรองความกลมเพื่อเลี่ยง Hot pixels
    daofind = DAOStarFinder(fwhm=4.0, threshold=5.0 * std, roundhi=0.5, roundlo=-0.5)
    sources = daofind(data - median)
    
    if sources is None or len(sources) < 10:
        print(f"❌ Not enough sources detected.")
        return None
        
    # ตัดขอบภาพ 30 พิกเซลเพื่อความแม่นยำ
    MARGIN = 30
    mask = ((sources['xcentroid'] > MARGIN) & (sources['xcentroid'] < w - MARGIN) &
            (sources['ycentroid'] > MARGIN) & (sources['ycentroid'] < h - MARGIN))
    sources = sources[mask]
    
    # เลือกดาวที่สว่างที่สุด 40 ดวง
    sources.sort('flux')
    sources.reverse()
    top_stars = sources[:40]
    
    stars = []
    for row in top_stars:
        stars.append((row['xcentroid'] * xbin, row['ycentroid'] * ybin))
        
    print(f"    Detected {len(sources)} stars. Using the brightest {len(stars)} stars.")

    # ── 3. ทำ Plate Solving ด้วย Astrometry ──
    print("[3] Running Astrometry.net Solver (Manual Index Injection)...")
    
    size_hint = astrometry.SizeHint(
        lower_arcsec_per_pixel=scale_low,
        upper_arcsec_per_pixel=scale_high,
    )
    
    position_hint = None
    if ra_center is not None and dec_center is not None:
        position_hint = astrometry.PositionHint(
            ra_deg=ra_center, dec_deg=dec_center, radius_deg=radius
        )

    # ── แก้ปัญหา KeyError: กวาดหาไฟล์ .fits ในโฟลเดอร์ 4100, 4200, 5200 โดยตรง ──
    index_files = []
    series_to_load = ['4100', '4200', '5200']
    
    for series in series_to_load:
        folder_path = os.path.join(astrometry_cache, series)
        found_fits = glob.glob(os.path.join(folder_path, "*.fits"))
        for f in found_fits:
            index_files.append(Path(f))
            
    if not index_files:
        print(f"❌ No index files (.fits) found in {astrometry_cache}. Please check paths.")
        return None

    print(f"    Loaded {len(index_files)} index files from local storage.")

    # รัน Solver โดยส่ง List ของ Path ไฟล์ตรงๆ
    with astrometry.Solver(index_files) as solver:
        solution = solver.solve(
            stars=stars,
            size_hint=size_hint,
            position_hint=position_hint,
            solution_parameters=astrometry.SolutionParameters(),
        )

    # ── 4. สรุปผลลัพธ์ ──
    if not solution.has_match():
        print("❌ Astrometry failed to find a WCS match. (Check your star detection or index files)")
        return None

    match = solution.best_match()
    print("✅ Success! Image Solved.")
    print(f"    Center RA : {match.center_ra_deg:.5f}°")
    print(f"    Center DEC: {match.center_dec_deg:.5f}°")
    print(f"    Pixel Scale: {match.scale_arcsec_per_pixel:.4f} arcsec/pixel")
    
    return match.wcs

# ==========================================
# ส่วนเรียกใช้งาน
# ==========================================
if __name__ == "__main__":
    # ระบุพาทไฟล์ HCM ของคุณ
    test_file = "/lustre/MSSP/sittipong/temp/hcam_reduction/data/data_run014_004.hcm" 
    
    if os.path.exists(test_file):
        # ใส่พิกัด Hint จากที่ได้บนเว็บเพื่อความเร็วสูงสุด
        wcs_result = solve_hcm_direct(
            hcm_file=test_file,
            astrometry_cache='/lustre/MSSP/sittipong/astrometry_cache',
            ra_center=322.499, 
            dec_center=-4.465,
            scale_low=0.8, 
            scale_high=1.1   # บีบสเกลให้ใกล้เคียง 0.904 ที่เว็บเจอ
        )
    else:
        print(f"❌ File not found: {test_file}")


[1] Reading file: /lustre/MSSP/sittipong/temp/hcam_reduction/data/data_run014_004.hcm (CCD: 1, Win: 1)
[2] Finding stars using photutils (DAOStarFinder)...
    Detected 93 stars. Using the brightest 40 stars.
[3] Running Astrometry.net Solver (Manual Index Injection)...
    Loaded 392 index files from local storage.
❌ Astrometry failed to find a WCS match. (Check your star detection or index files)
